#Guía 2: SQL

En esta guía usted deberá realizar consultas SQL en un servidor virtual con PostgreSQL que contiene
datos de películas extraídos de IMDb. El esquema de los datos es el siguiente:

* $\color{green}{\textbf{pelicula}}(\color{blue}{\underline{\text{nombre}}}, \color{blue}{\underline{\text{anho}}}, \color{blue}{\text{calificacion}})$
* $\color{green}{\textbf{actor}}(\color{blue}{\underline{\text{nombre}}}, \color{blue}{\text{genero}})$
* $\color{green}{\textbf{personaje}}(\color{blue}{\underline{\text{p_nombre}}}, \color{blue}{\underline{\text{p_anho}}},  \color{blue}{\underline{\text{a_nombre}}}, \color{blue}{\underline{\text{personaje}}})$

La tabla $\color{green}{\textbf{personaje}}$ usa llaves foráneas que hacen referencia a las tablas de $\color{green}{\textbf{actor}}(\color{blue}{\underline{\text{a_nombre}}})$ y $\color{green}{\textbf{pelicula}}(\color{blue}{\underline{\text{p_nombre}}}, \color{blue}{\underline{\text{p_anho}}})$.

Para iniciar el servidor virtual, instalar la base de datos postgres, y descargar los datos e importarlos, debe correr el siguiente bloque:



In [1]:
import subprocess, sys, os, sqlite3, urllib.request, pathlib

# Instalar dependencias si faltan
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ipython-sql', 'sqlalchemy'])

# Descargar el dump de PostgreSQL si no existe
dump_path = pathlib.Path('guia2.sql')
if not dump_path.exists():
    url = 'https://drive.google.com/uc?export=download&id=1g7UsolqIt5CatIUiWKQNiQ4sT6zsCZnk'
    urllib.request.urlretrieve(url, dump_path)
    print('Dump descargado')

# Parsear el dump PostgreSQL y cargar en SQLite
db_path = pathlib.Path('guia2.db')
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(str(db_path))
cur = conn.cursor()
cur.execute('CREATE TABLE pelicula (nombre TEXT NOT NULL, anho INTEGER NOT NULL, calificacion REAL, PRIMARY KEY (nombre, anho))')
cur.execute('CREATE TABLE actor (nombre TEXT NOT NULL PRIMARY KEY, genero TEXT NOT NULL)')
cur.execute('CREATE TABLE personaje (p_nombre TEXT NOT NULL, p_anho INTEGER NOT NULL, a_nombre TEXT NOT NULL, personaje TEXT NOT NULL, PRIMARY KEY (p_nombre, p_anho, a_nombre, personaje), FOREIGN KEY (p_nombre, p_anho) REFERENCES pelicula(nombre, anho), FOREIGN KEY (a_nombre) REFERENCES actor(nombre))')

lines = dump_path.read_text(encoding='utf-8').splitlines()
i, n = 0, len(lines)
while i < n:
    line = lines[i]
    if line.startswith('COPY guia.actor'):
        i += 1
        while i < n and lines[i] != '\\.':
            parts = lines[i].split('\t')
            cur.execute('INSERT INTO actor VALUES (?,?)', (parts[0], parts[1]))
            i += 1
    elif line.startswith('COPY guia.pelicula'):
        i += 1
        while i < n and lines[i] != '\\.':
            parts = lines[i].split('\t')
            cur.execute('INSERT INTO pelicula VALUES (?,?,?)', (parts[0], int(parts[1]), float(parts[2])))
            i += 1
    elif line.startswith('COPY guia.personaje'):
        i += 1
        while i < n and lines[i] != '\\.':
            parts = lines[i].split('\t')
            cur.execute('INSERT INTO personaje VALUES (?,?,?,?)', (parts[1], int(parts[2]), parts[0], parts[3]))
            i += 1
    i += 1

conn.commit()
conn.close()
print(f'Base de datos SQLite creada: {db_path}')

# Conectar ipython-sql a SQLite
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%config SqlMagic.feedback = False
%config SqlMagic.autopandas = True
%sql sqlite:///guia2.db

Base de datos SQLite creada: guia2.db


Ejecute la siguiente consulta para probar que todo ande bien:

In [2]:
%sql select * from pelicula;

 * sqlite:///guia2.db


,nombre,anho,calificacion
0,The Shawshank Redemption,1994,9.2
1,The Godfather,1972,9.2
2,The Godfather: Part II,1974,9.0
3,The Dark Knight,2008,8.9
4,12 Angry Men,1957,8.9
...,...,...,...
245,The King's Speech,2010,8.0
246,The Avengers,2012,8.0
247,Lagaan: Once Upon a Time in India,2001,8.0
248,Beauty and the Beast,1991,8.0


Para ejecutar consultas multi-lineas use el tag %%sql:

In [3]:
%%sql
SELECT * FROM
pelicula
where nombre LIKE '%terminator%';

 * sqlite:///guia2.db


,nombre,anho,calificacion
0,Terminator 2: Judgment Day,1991,8.5
1,The Terminator,1984,8.0


Ahora, debe diseñar las consultas que resuelvan las siguientes preguntas usando los operadores vistos en clases.

#SQL
## Pregunta 1
Las películas de los 80s, ordenadas por calificación de mayor a menor.

In [4]:
%%sql
SELECT *
FROM pelicula
WHERE anho >= 1980 AND anho < 1990
ORDER BY calificacion DESC;

 * sqlite:///guia2.db


,nombre,anho,calificacion
0,Star Wars: Episode V - The Empire Strikes Back,1980,8.7
1,Raiders of the Lost Ark,1981,8.5
2,Back to the Future,1985,8.5
3,Nuovo Cinema Paradiso,1988,8.4
4,The Shining,1980,8.4
5,Hotaru no haka,1988,8.4
6,Aliens,1986,8.4
7,Once Upon a Time in America,1984,8.4
8,Das Boot,1981,8.3
9,Star Wars: Episode VI - Return of the Jedi,1983,8.3


## Pregunta 2
Los nombres de los personajes que ha interpretado su actriz/actor favorito en los datos, ordenados por año.

In [5]:
%%sql
SELECT pe.personaje
FROM personaje pe
WHERE pe.a_nombre = 'Freeman, Morgan (I)'
ORDER BY pe.p_anho;

 * sqlite:///guia2.db


,personaje
0,Ned Logan
1,Ellis Boyd 'Red' Redding
2,Somerset
3,Eddie Scrap-Iron Dupris
4,Lucius Fox
5,Lucius Fox
6,Fox


# Pregunta 3
Las películas en las que participó su actriz/actor favorito en los datos, ordenadas por calificación de mayor a menor.

In [6]:
%%sql
SELECT DISTINCT p.nombre, p.anho, p.calificacion
FROM pelicula p
JOIN personaje pe ON p.nombre = pe.p_nombre AND p.anho = pe.p_anho
WHERE pe.a_nombre = 'Freeman, Morgan (I)'
ORDER BY p.calificacion DESC;

 * sqlite:///guia2.db


,nombre,anho,calificacion
0,The Shawshank Redemption,1994,9.2
1,The Dark Knight,2008,8.9
2,Se7en,1995,8.6
3,The Dark Knight Rises,2012,8.4
4,Batman Begins,2005,8.2
5,Unforgiven,1992,8.2
6,Million Dollar Baby,2004,8.1


## Pregunta 4
Los nombres de los personajes interpretados por mujeres, en películas de los 90s, con calificación mayor a 8,7.

In [7]:
%%sql
SELECT pe.personaje
FROM personaje pe
JOIN pelicula p ON p.nombre = pe.p_nombre AND p.anho = pe.p_anho
JOIN actor a ON a.nombre = pe.a_nombre
WHERE a.genero = 'F'
  AND p.anho >= 1990 AND p.anho < 2000
  AND p.calificacion > 8.7;

 * sqlite:///guia2.db


,personaje
0,Winston Wolfe's Girlfriend At Party
1,Fourth Man
2,Jody
3,Herself - Schindler Mourner
4,Ghetto Girl
...,...
79,Bonnie Dimmick
80,Brinnlitz Girl
81,Clara Sternberg
82,Plaszow Jewish Girl


## Pregunta 5
Las películas de la saga "The Lord of the Rings" (usando el prefijo de su nombre) ordenadas por calificación y por año.

In [8]:
%%sql
SELECT *
FROM pelicula
WHERE nombre LIKE 'The Lord of the Rings%'
ORDER BY calificacion DESC, anho;

 * sqlite:///guia2.db


,nombre,anho,calificacion
0,The Lord of the Rings: The Return of the King,2003,8.9
1,The Lord of the Rings: The Fellowship of the Ring,2001,8.8
2,The Lord of the Rings: The Two Towers,2002,8.7


## Pregunta 6
Los nombres de los actores que interpretan más de un personaje en la misma película.

In [9]:
%%sql
SELECT pe.a_nombre, pe.p_nombre, pe.p_anho, COUNT(*) AS num_personajes
FROM personaje pe
GROUP BY pe.a_nombre, pe.p_nombre, pe.p_anho
HAVING COUNT(*) > 1
ORDER BY pe.a_nombre;

 * sqlite:///guia2.db


,a_nombre,p_nombre,p_anho,num_personajes
0,"Abbott, Frankie",A Clockwork Orange,1971,2
1,"Abbou, Bernard",Les quatre cents coups,1959,2
2,"Adán, Pablo",El laberinto del fauno,2006,2
3,"Alazraqui, Carlos",Inside Out,2015,2
4,"Albertson, Jeff",The Dark Knight,2008,2
...,...,...,...,...
337,"Worsham, John",Forrest Gump,1994,2
338,"Yee, Mitchell (I)",The Dark Knight Rises,2012,3
339,"Yoshida, Rihoko",Kaze no tani no Naushika,1984,2
340,"Young, John (I)",Monty Python and the Holy Grail,1975,2


## Pregunta 7
Las películas en que actúan juntos Uma Thurman y Samuel L. Jackson.

In [10]:
%%sql
SELECT DISTINCT p.nombre, p.anho, p.calificacion
FROM pelicula p
JOIN personaje pe1 ON p.nombre = pe1.p_nombre AND p.anho = pe1.p_anho
JOIN personaje pe2 ON p.nombre = pe2.p_nombre AND p.anho = pe2.p_anho
WHERE pe1.a_nombre = 'Thurman, Uma'
  AND pe2.a_nombre = 'Jackson, Samuel L.';

 * sqlite:///guia2.db


,nombre,anho,calificacion
0,Pulp Fiction,1994,8.9


## Pregunta 8
Las películas en que actúa Uma Thurman y **no** Samuel L. Jackson.

In [11]:
%%sql
SELECT DISTINCT p.nombre, p.anho, p.calificacion
FROM pelicula p
JOIN personaje pe ON p.nombre = pe.p_nombre AND p.anho = pe.p_anho
WHERE pe.a_nombre = 'Thurman, Uma'
  AND NOT EXISTS (
    SELECT 1 FROM personaje pe2
    WHERE pe2.p_nombre = p.nombre
      AND pe2.p_anho = p.anho
      AND pe2.a_nombre = 'Jackson, Samuel L.'
  );

 * sqlite:///guia2.db


,nombre,anho,calificacion
0,Kaze no tani no Naushika,1984,8.1
1,Kill Bill: Vol. 1,2003,8.1


# PREGUNTA DESAFÍO (MUY COMPLEJA)

## Pregunta 9
La película con calificación más alta.

In [12]:
%%sql
SELECT *
FROM pelicula
WHERE calificacion = (SELECT MAX(calificacion) FROM pelicula);

 * sqlite:///guia2.db


,nombre,anho,calificacion
0,The Shawshank Redemption,1994,9.2
1,The Godfather,1972,9.2
